In [ ]:
import os
import pathlib
import pickle
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121
from sklearn.metrics import accuracy_score, classification_report, log_loss
from sklearn.model_selection import StratifiedShuffleSplit
from PIL import Image
import matplotlib.pyplot as plt

print('TensorFlow:', tf.__version__)
SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

2025-11-18 05:29:00.746916: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow: 2.20.0


In [ ]:
# Configuration – must match your training notebook
BASE_DIR = os.path.abspath('.')
DATA_SOURCE = 'folder'  # one of: 'folder', 'npy', 'pkl'
DATA_DIR = os.path.join(BASE_DIR, 'OCT', 'OCT')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT = 0.2
NUM_WORKERS = tf.data.AUTOTUNE
print('Train dir exists:', os.path.exists(TRAIN_DIR))

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')


def scan_folder(folder):
    class_names = sorted([d.name for d in pathlib.Path(folder).iterdir() if d.is_dir()])
    paths, labels = [], []
    for idx, cname in enumerate(class_names):
        cdir = pathlib.Path(folder) / cname
        for p in cdir.rglob('*'):
            if not p.is_file() or p.suffix.lower() not in IMG_EXTS:
                continue
            try:
                with Image.open(p) as im:
                    im.verify()
            except Exception:
                continue
            paths.append(str(p))
            labels.append(idx)
    return np.array(paths), np.array(labels, dtype=np.int64), class_names


def decode_and_resize(path):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    return image


def paths_labels_to_ds(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)

    def _map(path, label):
        img = decode_and_resize(path)
        return img, label

    ds = ds.map(_map, num_parallel_calls=NUM_WORKERS)
    ds = ds.batch(BATCH_SIZE).prefetch(NUM_WORKERS)
    try:
        ds = ds.ignore_errors()
    except AttributeError:
        ds = ds.apply(tf.data.experimental.ignore_errors())
    return ds


def load_data():
    if DATA_SOURCE == 'folder':
        tr_paths, tr_labels, class_names = scan_folder(TRAIN_DIR)
        splitter = StratifiedShuffleSplit(
            n_splits=1, test_size=VAL_SPLIT, random_state=SEED
        )
        tr_idx, val_idx = next(splitter.split(tr_paths, tr_labels))
        tr_paths_split, tr_labels_split = tr_paths[tr_idx], tr_labels[tr_idx]
        val_paths_split, val_labels_split = tr_paths[val_idx], tr_labels[val_idx]
        test_paths, test_labels, _ = scan_folder(TEST_DIR)

        train_ds = paths_labels_to_ds(tr_paths_split, tr_labels_split, shuffle=True)
        val_ds = paths_labels_to_ds(val_paths_split, val_labels_split, shuffle=False)
        test_ds = paths_labels_to_ds(test_paths, test_labels, shuffle=False)
        num_classes = len(class_names)
        return train_ds, val_ds, test_ds, class_names, num_classes, tr_labels_split

    elif DATA_SOURCE == 'npy':
        x_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
        y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
        x_test = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
        y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))
        splitter = StratifiedShuffleSplit(
            n_splits=1, test_size=VAL_SPLIT, random_state=SEED
        )
        idx_tr, idx_val = next(splitter.split(x_train, y_train))
        x_tr, y_tr = x_train[idx_tr], y_train[idx_tr]
        x_val, y_val = x_train[idx_val], y_train[idx_val]
        class_names = [str(i) for i in np.unique(y_train)]
        num_classes = len(class_names)

        def make_ds(x, y, training=False):
            ds = tf.data.Dataset.from_tensor_slices((x, y))
            if training:
                ds = ds.shuffle(4096, seed=SEED)
            ds = ds.batch(BATCH_SIZE).prefetch(NUM_WORKERS)
            return ds

        return (
            make_ds(x_tr, y_tr, True),
            make_ds(x_val, y_val),
            make_ds(x_test, y_test),
            class_names,
            num_classes,
            y_tr,
        )

    elif DATA_SOURCE == 'pkl':
        with open(os.path.join(DATA_DIR, 'data.pkl'), 'rb') as f:
            data = pickle.load(f)
        x_train, y_train = data['x_train'], data['y_train']
        x_test, y_test = data['x_test'], data['y_test']
        splitter = StratifiedShuffleSplit(
            n_splits=1, test_size=VAL_SPLIT, random_state=SEED
        )
        idx_tr, idx_val = next(splitter.split(x_train, y_train))
        x_tr, y_tr = x_train[idx_tr], y_train[idx_tr]
        x_val, y_val = x_train[idx_val], y_train[idx_val]
        class_names = [str(i) for i in np.unique(y_train)]
        num_classes = len(class_names)

        def make_ds(x, y, training=False):
            ds = tf.data.Dataset.from_tensor_slices((x, y))
            if training:
                ds = ds.shuffle(4096, seed=SEED)
            ds = ds.batch(BATCH_SIZE).prefetch(NUM_WORKERS)
            return ds

        return (
            make_ds(x_tr, y_tr, True),
            make_ds(x_val, y_val),
            make_ds(x_test, y_test),
            class_names,
            num_classes,
            y_tr,
        )
    else:
        raise ValueError('Unknown DATA_SOURCE')

train_ds_raw, val_ds_raw, test_ds_raw, class_names, num_classes, train_labels_arr = load_data()
num_classes, class_names

Train dir exists: True


I0000 00:00:1763443793.550663  457062 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 18091 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:21:00.0, compute capability: 8.6


(4, ['CNV', 'DME', 'DRUSEN', 'NORMAL'])

In [3]:
# Preprocessing (same as training)
preprocess_input = tf.keras.applications.densenet.preprocess_input


def prepare_ds(ds, training=False):
    def _map(x, y):
        x = tf.cast(x, tf.float32)
        if training:
            # For this metrics notebook we usually don't apply extra augmentation
            pass
        x = preprocess_input(x)
        return x, y

    ds = ds.map(_map, num_parallel_calls=NUM_WORKERS)
    if training:
        ds = ds.shuffle(1024, seed=SEED)
    return ds.prefetch(NUM_WORKERS)


# Datasets consistent with training
train_ds = prepare_ds(train_ds_raw, training=True)
val_ds = prepare_ds(val_ds_raw, training=False)
test_ds = prepare_ds(test_ds_raw, training=False)

# A non-augmented train dataset for clean train accuracy/log-loss
train_eval_ds = prepare_ds(train_ds_raw, training=False)

In [4]:
def evaluate_model(model, ds, class_names):
    # Collect true labels
    y_true = np.concatenate([y.numpy() for _, y in ds], axis=0)
    # Predicted probabilities
    proba = model.predict(ds, verbose=0)
    y_pred = np.argmax(proba, axis=1)
    acc = accuracy_score(y_true, y_pred)
    ll = log_loss(y_true, proba, labels=list(range(len(class_names))))
    report = classification_report(
        y_true, y_pred, target_names=class_names, digits=4
    )
    return {
        'accuracy': acc,
        'log_loss': ll,
        'report': report,
        'y_true': y_true,
        'y_pred': y_pred,
    }


def print_phase_split_results(phase_name, split_name, res):
    print(f"{phase_name} {split_name} -> acc: {res['accuracy']:.3f} log_loss: {res['log_loss']}")
    print(res['report'])
    print()

In [5]:
ckpt_p1 = os.path.join(BASE_DIR, 'densenet_phase1_best.keras')
ckpt_p2 = os.path.join(BASE_DIR, 'densenet_phase2_best.keras')

print('Phase 1 checkpoint:', ckpt_p1, 'exists:', os.path.exists(ckpt_p1))
print('Phase 2 checkpoint:', ckpt_p2, 'exists:', os.path.exists(ckpt_p2))

m1 = keras.models.load_model(ckpt_p1)
m2 = keras.models.load_model(ckpt_p2)

Phase 1 checkpoint: /home/putta.abhiram.23cse/DIP - Retinal Disease classification/dip/densenet_phase1_best.keras exists: True
Phase 2 checkpoint: /home/putta.abhiram.23cse/DIP - Retinal Disease classification/dip/densenet_phase2_best.keras exists: True


In [6]:
# Phase 1 – evaluate on train / val / test

res1_train = evaluate_model(m1, train_eval_ds, class_names)
res1_val = evaluate_model(m1, val_ds, class_names)
res1_test = evaluate_model(m1, test_ds, class_names)

print_phase_split_results('Phase 1', 'Train', res1_train)
print_phase_split_results('Phase 1', 'Val', res1_val)
print_phase_split_results('Phase 1', 'Test', res1_test)

2025-11-18 05:30:45.943815: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-11-18 05:30:49.748893: I external/local_xla/xla/service/service.cc:163] XLA service 0x708e20001d80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-18 05:30:49.748921: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3090, Compute Capability 8.6
2025-11-18 05:30:49.920457: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-18 05:30:50.870809: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91200
2025-11-18 05:30:51.248497: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3301', 8 b

Phase 1 Train -> acc: 0.248 log_loss: 2.159924692381198
              precision    recall  f1-score   support

         CNV     0.2450    0.2373    0.2411     40294
         DME     0.2495    0.2537    0.2516     40912
      DRUSEN     0.2497    0.1767    0.2069     40912
      NORMAL     0.2480    0.3240    0.2809     40912

    accuracy                         0.2480    163030
   macro avg     0.2481    0.2479    0.2451    163030
weighted avg     0.2481    0.2480    0.2452    163030


Phase 1 Val -> acc: 0.687 log_loss: 0.8047410294118115
              precision    recall  f1-score   support

         CNV     0.7543    0.7342    0.7441     10074
         DME     0.6382    0.6500    0.6440     10228
      DRUSEN     0.6607    0.4687    0.5484     10228
      NORMAL     0.6898    0.8957    0.7794     10228

    accuracy                         0.6870     40758
   macro avg     0.6858    0.6871    0.6790     40758
weighted avg     0.6855    0.6870    0.6787     40758


Phase 1 Test -> a

/home/putta.abhiram.23cse/DIP - Retinal Disease classification/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [7]:
# Phase 2 – evaluate on train / val / test

res2_train = evaluate_model(m2, train_eval_ds, class_names)
res2_val = evaluate_model(m2, val_ds, class_names)
res2_test = evaluate_model(m2, test_ds, class_names)

print_phase_split_results('Phase 2', 'Train', res2_train)
print_phase_split_results('Phase 2', 'Val', res2_val)
print_phase_split_results('Phase 2', 'Test', res2_test)

/home/putta.abhiram.23cse/DIP - Retinal Disease classification/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
2025-11-18 05:36:41.950650: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/home/putta.abhiram.23cse/DIP - Retinal Disease classification/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


Phase 2 Train -> acc: 0.247 log_loss: 10.191169497367342
              precision    recall  f1-score   support

         CNV     0.2462    0.2366    0.2413     40294
         DME     0.2459    0.2134    0.2285     40912
      DRUSEN     0.2484    0.2489    0.2486     40912
      NORMAL     0.2477    0.2895    0.2670     40912

    accuracy                         0.2471    163030
   macro avg     0.2470    0.2471    0.2463    163030
weighted avg     0.2471    0.2471    0.2464    163030


Phase 2 Val -> acc: 0.885 log_loss: 0.6629587457275101
              precision    recall  f1-score   support

         CNV     0.9154    0.8814    0.8980     10074
         DME     0.9423    0.8155    0.8743     10228
      DRUSEN     0.8439    0.8502    0.8470     10228
      NORMAL     0.8524    0.9919    0.9169     10228

    accuracy                         0.8848     40758
   macro avg     0.8885    0.8847    0.8841     40758
weighted avg     0.8884    0.8848    0.8840     40758


Phase 2 Test -> 

/home/putta.abhiram.23cse/DIP - Retinal Disease classification/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
